# Declarative Pipelines: `@dp` vs `@dlt` — A Complete Guide

This notebook provides a deep-dive comparison of the two Python APIs for building Lakeflow Spark Declarative Pipelines (SDP). Both define the same underlying pipeline engine — the difference is **how you write the code**.

| | Classic API | Modern API |
|---|---|---|
| **Import** | `import dlt` | `from pyspark import pipelines as dp` |
| **Introduced** | With Delta Live Tables (2021) | DBR 16.0+ (2024), open-source standard |
| **Alias used in this notebook** | `dlt` | `dp` |

## 📋 What We'll Cover

1. **Origins & Philosophy** — why two APIs exist
2. **Import & Setup** — how each API is loaded
3. **Decorator Mapping** — complete side-by-side reference
4. **Reading Pipeline Tables** — the most important practical difference
5. **Streaming Tables** — bronze ingestion side-by-side
6. **Materialized Views** — where `@dp` shines with a dedicated decorator
7. **Temporary & Persistent Views** — naming and behaviour differences
8. **Expectations (Data Quality)** — identical syntax, different namespace
9. **Auto CDC (Change Data Capture)** — renamed and improved in `@dp`
10. **Sinks** — exclusive to `@dp`
11. **Benefits Comparison** — what each API does better
12. **Decision Framework** — when to choose which
13. **Migration Guide** — converting `@dlt` to `@dp`
14. **This Training's Pipeline Files** — see both in action

## 1. Origins & Philosophy

### The `@dlt` API (Classic)

When Databricks launched **Delta Live Tables (DLT)** in 2021, it shipped with a proprietary Python module called `dlt`. This was the **only** way to define pipeline datasets in Python. It introduced:
* Decorator-based table definitions (`@dlt.table`)
* Built-in data quality expectations (`@dlt.expect`)
* Custom read functions (`dlt.read()`, `dlt.read_stream()`)
* The `LIVE.` prefix for inter-table references in SQL

The API was tightly coupled to the Databricks platform — code written with `import dlt` could **only** run inside a Databricks pipeline.

### The `@dp` API (Modern)

In 2024 (DBR 16.0+), Databricks contributed the declarative pipeline framework to **Apache Spark as an open-source standard**. The new API lives under `pyspark.pipelines` and is designed to be:
* **Portable** — works on any Spark distribution that adopts the standard
* **Explicit** — separate decorators for streaming tables vs materialized views
* **Aligned with Spark** — uses standard `spark.read.table()` / `spark.readStream.table()` instead of custom readers
* **Forward-looking** — new features (sinks, improved CDC) are added here first

### Key philosophical difference

| | `@dlt` (Classic) | `@dp` (Modern) |
|---|---|---|
| **Design goal** | Databricks-native convenience | Open-source Spark standard |
| **Coupling** | Tightly coupled to Databricks | Portable across Spark distributions |
| **Reads** | Custom `dlt.read()` / `dlt.read_stream()` | Standard `spark.read.table()` / `spark.readStream.table()` |
| **Naming** | Overloaded `@dlt.table` for both ST and MV | Explicit `@dp.table()` (ST) vs `@dp.materialized_view()` (MV) |
| **New features** | No new features being added | All new features land here first |

## 2. Import & Setup

### Classic `@dlt`
```python
import dlt
from pyspark.sql.functions import col, upper, trim
```
* The `dlt` module is auto-available inside any SDP pipeline — no pip install needed.
* `spark` is also pre-available (no need to create a SparkSession).

### Modern `@dp`
```python
from pyspark import pipelines as dp
from pyspark.sql.functions import col, upper, trim
```
* `pyspark.pipelines` is available from **DBR 16.0+** inside SDP pipelines.
* Same `spark` session is pre-available.

### Side-by-side

```python
# Classic                          # Modern
import dlt                          from pyspark import pipelines as dp
```

> **Rule of thumb:** For new code, always use `from pyspark import pipelines as dp`. Keep `import dlt` in existing pipelines that already use it — there is no urgency to migrate working code.

## 3. Complete Decorator & Function Mapping

This is the **full reference** of how every `dlt` construct maps to its `dp` equivalent.

### Table & View Definitions

| Purpose | `@dlt` (Classic) | `@dp` (Modern) | Notes |
|---|---|---|---|
| Streaming table | `@dlt.table()` + returns `readStream` | `@dp.table()` + returns `readStream` | Identical behaviour |
| Materialized view | `@dlt.table()` + returns batch `read` | **`@dp.materialized_view()`** | `dp` has a **dedicated decorator** — clearer intent |
| Temporary view | `@dlt.view()` | `@dp.temporary_view()` | More descriptive name in `dp` |
| Persistent UC view | SQL only (`CREATE VIEW`) | SQL only (`CREATE VIEW`) | Neither API supports this in Python |

### Data Quality (Expectations)

| Purpose | `@dlt` (Classic) | `@dp` (Modern) |
|---|---|---|
| Warn (log, keep row) | `@dlt.expect(name, expr)` | `@dp.expect(name, expr)` |
| Drop failing row | `@dlt.expect_or_drop(name, expr)` | `@dp.expect_or_drop(name, expr)` |
| Fail the pipeline | `@dlt.expect_or_fail(name, expr)` | `@dp.expect_or_fail(name, expr)` |
| Multiple rules (warn) | `@dlt.expect_all(rules_dict)` | `@dp.expect_all(rules_dict)` |
| Multiple rules (drop) | `@dlt.expect_all_or_drop(rules_dict)` | `@dp.expect_all_or_drop(rules_dict)` |
| Multiple rules (fail) | `@dlt.expect_all_or_fail(rules_dict)` | `@dp.expect_all_or_fail(rules_dict)` |

### Reads

| Purpose | `@dlt` (Classic) | `@dp` (Modern) |
|---|---|---|
| Streaming read | `dlt.read_stream("table")` | `spark.readStream.table("table")` |
| Batch read | `dlt.read("table")` | `spark.read.table("table")` |

### Advanced Features

| Purpose | `@dlt` (Classic) | `@dp` (Modern) |
|---|---|---|
| Create streaming table (programmatic) | `dlt.create_streaming_table()` | `dp.create_streaming_table()` |
| Auto CDC (change data capture) | `dlt.apply_changes()` (legacy name) | `dp.create_auto_cdc_flow()` |
| Auto CDC from snapshots | `dlt.apply_changes_from_snapshot()` | `dp.create_auto_cdc_from_snapshot_flow()` |
| Append flow | `dlt.create_flow()` | `dp.append_flow()` |
| **Sinks (Delta/Kafka/custom)** | **Not available** | `dp.create_sink()` |
| **ForEachBatch sink** | **Not available** | `@dp.foreach_batch_sink()` |
| `LIVE.` prefix | Supported but deprecated | **Not supported — use table names directly** |

## 4. Reading Pipeline Tables — The Most Important Difference

This is the **single biggest practical difference** between the two APIs and the main reason to prefer `@dp`.

### Classic `@dlt` — custom readers
```python
import dlt

@dlt.table(name="customer_silver")
def customer_silver():
    # dlt.read_stream() is a custom, Databricks-only function
    return dlt.read_stream("customer_bronze").filter("c_custkey IS NOT NULL")
```

**Problems:**
* `dlt.read_stream()` only works inside a pipeline — you **cannot** test this function in a notebook
* The function signature is not standard Spark — new team members must learn a custom API
* If you refactor the function to run outside a pipeline (e.g. for unit testing), you must rewrite the reads

### Modern `@dp` — standard Spark readers
```python
from pyspark import pipelines as dp

@dp.table(name="customer_silver")
def customer_silver():
    # spark.readStream.table() is standard Spark — works everywhere
    return spark.readStream.table("customer_bronze").filter("c_custkey IS NOT NULL")
```

**Benefits:**
* `spark.readStream.table()` is **standard PySpark** — same API you use in notebooks, jobs, and streaming apps
* Code is **testable** outside a pipeline with a simple SparkSession
* No lock-in to a Databricks-specific reader function
* IntelliSense / autocomplete works better since `spark` is a well-known object

### Summary

| | `dlt.read_stream()` | `spark.readStream.table()` |
|---|---|---|
| Works inside pipeline | Yes | Yes |
| Works in a notebook | No | Yes |
| Works in unit tests | No | Yes |
| Standard PySpark | No | Yes |
| IDE autocomplete | Limited | Full |

## 5. Streaming Tables — Bronze Ingestion Side-by-Side

Streaming tables are the **most common dataset type** in SDP. The definition is nearly identical in both APIs.

### Classic `@dlt`
```python
import dlt
from pyspark.sql.functions import current_timestamp

@dlt.expect("no rescued data", "_rescued_data IS NULL")
@dlt.expect_or_drop("valid_key", "c_custkey IS NOT NULL")
@dlt.table(
    name="customer_bronze",
    comment="Raw customer records via Auto Loader",
    table_properties={"quality": "bronze"},
)
def customer_bronze():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema("c_custkey INT, c_name STRING, c_acctbal DOUBLE")
        .load("abfss://inboundfiles@storage.dfs.core.windows.net/customer")
        .withColumn("_ingest_ts", current_timestamp())
    )
```

### Modern `@dp`
```python
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp

@dp.expect("no rescued data", "_rescued_data IS NULL")
@dp.expect_or_drop("valid_key", "c_custkey IS NOT NULL")
@dp.table(
    name="customer_bronze",
    comment="Raw customer records via Auto Loader",
    table_properties={"quality": "bronze"},
)
def customer_bronze():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema("c_custkey INT, c_name STRING, c_acctbal DOUBLE")
        .load("abfss://inboundfiles@storage.dfs.core.windows.net/customer")
        .withColumn("_ingest_ts", current_timestamp())
    )
```

### What changed?
* `import dlt` → `from pyspark import pipelines as dp`
* `@dlt.table` → `@dp.table`
* `@dlt.expect*` → `@dp.expect*`
* **Everything else is identical** — the Auto Loader read, schema, options, and DataFrame transformations are standard Spark and don't change.

## 6. Materialized Views — Where `@dp` Shines

This is where the APIs diverge most. In `@dlt`, there is **no dedicated materialized view decorator** — you use the same `@dlt.table` and the engine infers the type from whether you return a streaming or batch DataFrame.

### Classic `@dlt` — implicit type
```python
import dlt

@dlt.table(
    name="customer_gold",
    comment="Customer aggregation by segment",
)
def customer_gold():
    # Returning a batch DataFrame makes this a materialized view.
    # But nothing in the decorator tells you that — you have to read the code.
    return (
        dlt.read("customer_silver")
        .groupBy("c_mktsegment")
        .agg({"c_acctbal": "sum", "c_custkey": "count"})
    )
```

**Problem:** Looking at `@dlt.table`, you can't tell if this is a streaming table or a materialized view without reading the function body.

### Modern `@dp` — explicit type
```python
from pyspark import pipelines as dp

@dp.materialized_view(                    # ← CLEAR INTENT
    name="customer_gold",
    comment="Customer aggregation by segment",
)
def customer_gold():
    return (
        spark.read.table("customer_silver")
        .groupBy("c_mktsegment")
        .agg({"c_acctbal": "sum", "c_custkey": "count"})
    )
```

**Benefits:**
* `@dp.materialized_view()` makes the **intent explicit** at the decorator level
* Code reviewers instantly know this is a full-recompute dataset
* No ambiguity about whether the function should return streaming or batch

### Comparison

| | `@dlt` | `@dp` |
|---|---|---|
| Decorator | `@dlt.table` (same for ST and MV) | `@dp.materialized_view()` (dedicated) |
| Read method | `dlt.read("table")` | `spark.read.table("table")` |
| Type clarity | Implicit — inferred from DataFrame type | **Explicit — declared in the decorator** |
| Code review | Must read body to determine type | Decorator tells you immediately |

## 7. Temporary & Persistent Views

### Temporary views (pipeline-scoped, not persisted)

Used for intermediate logic that doesn't need to be stored or queried outside the pipeline.

```python
# Classic                                    # Modern
@dlt.view()                                  @dp.temporary_view()
def filtered_customers():                    def filtered_customers():
    return dlt.read("customer_bronze")           return spark.read.table("customer_bronze")
        .filter("c_acctbal > 0")                     .filter("c_acctbal > 0")
```

| | `@dlt` | `@dp` |
|---|---|---|
| Decorator | `@dlt.view()` | `@dp.temporary_view()` |
| Name clarity | `view` could mean many things | `temporary_view` is unambiguous |

### Persistent views (published to Unity Catalog)

Persistent views are **SQL-only** in both APIs:
```sql
CREATE VIEW customer_active AS
SELECT * FROM customer_silver WHERE c_acctbal > 0;
```
Neither `@dlt` nor `@dp` supports persistent views in Python.

## 8. Expectations (Data Quality)

Expectations are functionally **identical** in both APIs — the only difference is the namespace prefix.

### Single expectation
```python
# Classic                                          # Modern
@dlt.expect("valid_key", "id IS NOT NULL")         @dp.expect("valid_key", "id IS NOT NULL")
@dlt.expect_or_drop("no_null", "name != ''")       @dp.expect_or_drop("no_null", "name != ''")
@dlt.expect_or_fail("positive", "amount > 0")      @dp.expect_or_fail("positive", "amount > 0")
```

### Multiple expectations (rules dict)
```python
rules = {
    "valid_key":    "c_custkey IS NOT NULL",
    "valid_name":   "c_name IS NOT NULL AND c_name != ''",
    "valid_segment": "c_mktsegment IS NOT NULL",
}

# Classic                                    # Modern
@dlt.expect_all(rules)                       @dp.expect_all(rules)           # warn
@dlt.expect_all_or_drop(rules)               @dp.expect_all_or_drop(rules)   # drop
@dlt.expect_all_or_fail(rules)               @dp.expect_all_or_fail(rules)   # fail
```

### What's the same?
* All three enforcement modes: **warn** (log), **drop** (remove row), **fail** (stop pipeline)
* Pass/fail metrics recorded in the pipeline event log
* Can be stacked (multiple decorators on one function)

### What's different?
Only the import — `dlt.expect` vs `dp.expect`. The behaviour, SQL expressions, and metrics are identical.

## 9. Auto CDC (Change Data Capture)

Auto CDC lets you track changes (inserts, updates, deletes) from a source into a target streaming table, with optional SCD Type 1 or Type 2 history.

### Classic `@dlt` — `apply_changes` (legacy name)
```python
import dlt

dlt.create_streaming_table("customers_cdc")

dlt.apply_changes(
    target="customers_cdc",
    source="raw_events",
    keys=["customer_id"],
    sequence_by="event_timestamp",
    stored_as_scd_type=2,
)
```

### Modern `@dp` — `create_auto_cdc_flow` (clearer name)
```python
from pyspark import pipelines as dp

dp.create_streaming_table("customers_cdc")

dp.create_auto_cdc_flow(
    target="customers_cdc",
    source="raw_events",
    keys=["customer_id"],
    sequence_by="event_timestamp",
    stored_as_scd_type=2,
)
```

### What changed?

| | `@dlt` | `@dp` |
|---|---|---|
| Function name | `dlt.apply_changes()` | `dp.create_auto_cdc_flow()` |
| From snapshots | `dlt.apply_changes_from_snapshot()` | `dp.create_auto_cdc_from_snapshot_flow()` |
| Naming clarity | "apply changes" is vague | "auto CDC flow" describes exactly what it does |
| Parameters | Same | Same |
| Behaviour | Identical | Identical |

## 10. Sinks — Exclusive to `@dp`

Sinks are **only available in the `@dp` API** and are one of the strongest reasons to choose it. A sink writes pipeline data to an **external target** that the pipeline does not own.

### Use cases for sinks
* Write to a **pre-existing Delta table** managed by another team
* Publish events to **Kafka / Event Hubs**
* Write to a **custom destination** (JDBC, REST API, etc.)

### Delta sink example
```python
from pyspark import pipelines as dp

# Create a sink that writes to an external Delta table
dp.create_sink(
    name="external_customer_sink",
    format="delta",
    table_name="shared_catalog.reporting.customers",
)

# Feed data into the sink with an append flow
@dp.append_flow(target="external_customer_sink")
def feed_external_customers():
    return spark.readStream.table("customer_silver")
```

### ForEachBatch sink example
```python
from pyspark import pipelines as dp

@dp.foreach_batch_sink(name="custom_sink")
def custom_writer(batch_df, batch_id):
    # Custom logic per micro-batch — e.g., call a REST API, merge into a table
    batch_df.write.format("delta").mode("append").saveAsTable("target_table")

@dp.append_flow(target="custom_sink")
def feed_custom():
    return spark.readStream.table("customer_silver")
```

### Why this matters

Without sinks, if you need to write to a table your pipeline doesn't own, you'd need a separate Spark job or notebook. Sinks keep everything **inside the pipeline graph** — with lineage, monitoring, and retry handling.

| Feature | `@dlt` | `@dp` |
|---|---|---|
| Delta sink | Not available | `dp.create_sink(format="delta")` |
| Kafka sink | Not available | `dp.create_sink(format="kafka")` |
| ForEachBatch sink | Not available | `@dp.foreach_batch_sink()` |
| Custom destinations | Not available | Supported via ForEachBatch |

## 11. Benefits Comparison

### Benefits of `@dp` (Modern API)

| Benefit | Detail |
|---|---|
| **Portability** | Code runs on any Spark distribution with the declarative pipelines standard, not just Databricks |
| **Standard reads** | `spark.read.table()` / `spark.readStream.table()` — same API as notebooks, jobs, and streaming apps |
| **Testability** | Functions can be unit-tested outside a pipeline with a standard SparkSession |
| **Explicit types** | `@dp.table()` vs `@dp.materialized_view()` — no ambiguity about dataset type |
| **Sinks** | Write to external tables, Kafka, and custom destinations from within the pipeline |
| **Clearer naming** | `create_auto_cdc_flow` is more descriptive than `apply_changes` |
| **Future-proof** | All new features land in `dp` first; `dlt` receives no new capabilities |
| **IDE support** | Better autocomplete since `spark` is a well-known object |

### Benefits of `@dlt` (Classic API)

| Benefit | Detail |
|---|---|
| **Mature ecosystem** | Vast majority of blog posts, Stack Overflow answers, and tutorials use `dlt` |
| **Simpler migration** | Existing pipelines already use it — no rewrite needed |
| **Shorter read syntax** | `dlt.read_stream("t")` is slightly shorter than `spark.readStream.table("t")` |
| **Familiarity** | Teams already trained on DLT documentation know this API |

### What's identical in both

* Pipeline execution engine (same DAG, same checkpoints, same retries)
* Data quality expectations (same syntax, same enforcement, same metrics)
* Auto Loader integration (same `cloudFiles` format and options)
* Pipeline UI (same graph, same monitoring, same event log)
* Performance (both compile to the same physical execution plan)
* SQL datasets (both support SQL `CREATE STREAMING TABLE` / `CREATE MATERIALIZED VIEW`)
* Unity Catalog publishing (same catalog/schema configuration)

## 12. Decision Framework — When to Choose Which

```
Are you writing a NEW pipeline from scratch?
├─ YES ───────────────────────────► Use @dp
│
└─ NO ── Do you have an existing @dlt pipeline?
        ├─ YES ── Is it working fine?
        │       ├─ YES ──────────────► Keep @dlt (no urgency to migrate)
        │       └─ NO / needs new features ► Migrate to @dp
        │
        └─ NO ─────────────────────► Use @dp
```

### Quick decision table

| Scenario | Use | Why |
|---|---|---|
| New pipeline from scratch | `@dp` | Forward-looking, all new features |
| Existing `@dlt` pipeline, working fine | `@dlt` | Don't fix what isn't broken |
| Need sinks (external writes) | `@dp` | Sinks are exclusive to `@dp` |
| Need explicit MV decorator | `@dp` | `@dp.materialized_view()` |
| Want unit-testable code | `@dp` | Standard `spark.read.table()` |
| Team trained on old DLT docs | `@dlt` | Familiarity reduces errors |
| Migrating to open-source Spark | `@dp` | Portable across distributions |
| Mixed team (some know dlt, some don't) | `@dp` | Standard PySpark is universal |
| Following an old tutorial/blog | `@dlt` | Match the tutorial's imports |

## 13. Migration Guide — Converting `@dlt` to `@dp`

Migrating is straightforward because both APIs produce the same pipeline engine. Here is a step-by-step checklist.

### Step 1: Change the import
```python
# Before
import dlt

# After
from pyspark import pipelines as dp
```

### Step 2: Replace decorators
| Find | Replace with |
|---|---|
| `@dlt.table(` | `@dp.table(` (for streaming tables) |
| `@dlt.table(` + batch read | `@dp.materialized_view(` |
| `@dlt.view(` | `@dp.temporary_view(` |
| `@dlt.expect(` | `@dp.expect(` |
| `@dlt.expect_or_drop(` | `@dp.expect_or_drop(` |
| `@dlt.expect_or_fail(` | `@dp.expect_or_fail(` |
| `@dlt.expect_all(` | `@dp.expect_all(` |
| `@dlt.expect_all_or_drop(` | `@dp.expect_all_or_drop(` |
| `@dlt.expect_all_or_fail(` | `@dp.expect_all_or_fail(` |

### Step 3: Replace reads
| Find | Replace with |
|---|---|
| `dlt.read_stream("table")` | `spark.readStream.table("table")` |
| `dlt.read("table")` | `spark.read.table("table")` |

### Step 4: Replace advanced functions
| Find | Replace with |
|---|---|
| `dlt.create_streaming_table(` | `dp.create_streaming_table(` |
| `dlt.apply_changes(` | `dp.create_auto_cdc_flow(` |
| `dlt.apply_changes_from_snapshot(` | `dp.create_auto_cdc_from_snapshot_flow(` |
| `dlt.create_flow(` | `dp.append_flow(` |

### Step 5: Remove `LIVE.` prefixes
```python
# Before
df = spark.read.table("LIVE.customer_bronze")

# After
df = spark.read.table("customer_bronze")
```

### Step 6: Validate
* Do a **dry run** of the pipeline to confirm all references resolve.
* Run a normal update and verify data output matches before-migration results.

> **Important:** You do **not** need to drop and recreate tables. The migration is purely a code change — the underlying Delta tables, checkpoints, and pipeline state are unaffected.

## 14. This Training's Pipeline Files

This training repo includes **both versions** of the same config-driven pipeline so you can compare them directly.

### Files

| File | API | Description |
|---|---|---|
| `sdp_pipeline_config.json` | (shared) | All source paths, schemas, table names, expectations, and transforms for customer + orders |
| `sdp_customer_pipeline.py` | `@dlt` | Classic API version — uses `import dlt`, `dlt.read_stream()` |
| `sdp_customer_pipeline_dp.py` | `@dp` | Modern API version — uses `from pyspark import pipelines as dp`, `spark.readStream.table()` |

### What to look for when comparing

1. **Line 17** — the import statement (`import dlt` vs `from pyspark import pipelines as dp`)
2. **Expectation helpers** — `dlt.expect()` vs `dp.expect()` in the `_apply_expectations` function
3. **Silver read** — `dlt.read_stream(bronze_name)` vs `spark.readStream.table(bronze_name)`
4. **Everything else** — identical! Config loading, Auto Loader options, transforms, schema enforcement

### Try it yourself

1. Create a pipeline in the UI
2. Add **one** of the Python files as source code (not both!)
3. Set catalog=`one_dbx_training`, schema=`default`
4. Run the pipeline and observe 4 tables: `customer_bronze`, `customer_silver`, `orders_bronze`, `orders_silver`
5. Swap the source file to the other version and re-run — same results, different code style

---

> **Remember:** `@dp` and `@dlt` are two **syntaxes** for the same engine. The pipeline doesn't care which API you use — it compiles to the same execution plan. Choose `@dp` for new work, keep `@dlt` for existing pipelines.